第一题

In [1]:
import pandas as pd
import numpy as np

# 加载数据和计算指标
df_supply = pd.read_excel('附件1.xlsx', sheet_name='供应商的供货量（m³）')
df_order = pd.read_excel('附件1.xlsx', sheet_name='企业的订货量（m³）')
df_order_clean = df_order[df_order['供应商ID'].isin(df_supply['供应商ID'].unique())].copy()
week_columns = [col for col in df_supply.columns if col.startswith('W')]

# 创建结果数据框
results = pd.DataFrame()
results['供应商ID'] = df_supply['供应商ID']
results['材料分类'] = df_supply['材料分类']

# 计算所有8个指标
material_coefficients = {'A': 1/0.6, 'B': 1/0.66, 'C': 1/0.72}
results['材料系数'] = results['材料分类'].map(material_coefficients)
results['总供货量'] = df_supply[week_columns].sum(axis=1)
results['等效总供货量'] = results['总供货量'] * results['材料系数']
results['总预定量'] = df_order_clean[week_columns].sum(axis=1)
results['到货率'] = np.where(results['总预定量'] > 0, results['总供货量'] / results['总预定量'], 0)

# 平均供货强度
supply_weeks_count = (df_supply[week_columns] > 0).sum(axis=1)
results['平均供货强度'] = np.where(supply_weeks_count > 0, results['总供货量'] / supply_weeks_count, 0)

# 其他指标计算
supply_response_rates = []
supply_sufficiency_rates = []
cv_values = []
max_continuous_weeks = []

for i in range(len(df_supply)):
    supply_data = df_supply.iloc[i][week_columns].values
    order_data = df_order_clean.iloc[i][week_columns].values

    # 供货响应率
    has_order = order_data > 0
    has_supply_and_order = (supply_data > 0) & has_order
    total_order_weeks = np.sum(has_order)
    response_rate = np.sum(has_supply_and_order) / total_order_weeks if total_order_weeks > 0 else 0

    # 供货充足率
    sufficiency_ratios = []
    for j in range(len(week_columns)):
        if order_data[j] > 0:
            ratio = supply_data[j] / order_data[j]
            sufficiency_ratios.append(min(ratio, 2.0))
    sufficiency_rate = np.mean(sufficiency_ratios) if sufficiency_ratios else 0

    # 供货变异系数
    valid_supply = supply_data[supply_data > 0]
    cv = np.std(valid_supply) / np.mean(valid_supply) if len(valid_supply) >= 2 else 0

    # 供货持续性
    continuous_count = 0
    max_continuous = 0
    in_continuous = False
    for j in range(len(week_columns)):
        if has_order[j]:
            if supply_data[j] > 0:
                if in_continuous:
                    continuous_count += 1
                else:
                    continuous_count = 1
                    in_continuous = True
                max_continuous = max(max_continuous, continuous_count)
            else:
                in_continuous = False
                continuous_count = 0

    supply_response_rates.append(response_rate)
    supply_sufficiency_rates.append(sufficiency_rate)
    cv_values.append(cv)
    max_continuous_weeks.append(max_continuous)

results['供货响应率'] = supply_response_rates
results['供货充足率'] = supply_sufficiency_rates
results['供货变异系数'] = cv_values
results['供货持续性'] = max_continuous_weeks

# TOPSIS模型实现
topsis_indicators = ['等效总供货量', '总预定量', '到货率', '平均供货强度', '供货响应率', '供货充足率', '供货持续性',
                     '供货变异系数']
indicator_directions = ['positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'negative']

# 提取指标数据
X = results[topsis_indicators].values

# 数据标准化
n, m = X.shape
X_normalized = np.zeros((n, m))

for j in range(m):
    if indicator_directions[j] == 'positive':
        # 正向指标标准化
        X_normalized[:, j] = X[:, j] / np.sqrt(np.sum(X[:, j] ** 2))
    else:
        # 负向指标标准化：先正向化，再标准化
        # 正向化：max - x
        max_val = np.max(X[:, j])
        X_positive = max_val - X[:, j]
        X_normalized[:, j] = X_positive / np.sqrt(np.sum(X_positive ** 2))

# 确定理想解和负理想解
ideal_best = np.max(X_normalized, axis=0)
ideal_worst = np.min(X_normalized, axis=0)

# 计算距离
d_best = np.sqrt(np.sum((X_normalized - ideal_best) ** 2, axis=1))
d_worst = np.sqrt(np.sum((X_normalized - ideal_worst) ** 2, axis=1))

# 计算综合评价指数
closeness = d_worst / (d_best + d_worst)

# 添加到结果中
results['TOPSIS综合得分'] = closeness
results['排名'] = results['TOPSIS综合得分'].rank(ascending=False, method='min').astype(int)

# 显示排名前10的供应商
print("TOPSIS模型分析结果 - 排名前10的供应商:")
top_10 = results.sort_values('TOPSIS综合得分', ascending=False).head(10)
print(top_10[['供应商ID', '材料分类', 'TOPSIS综合得分', '排名']])

# 显示50家最重要供应商的统计信息
top_50 = results.sort_values('TOPSIS综合得分', ascending=False).head(50)
print(f"\n前50家最重要供应商材料分类分布:")
print(top_50['材料分类'].value_counts())

print(f"\n前50家供应商TOPSIS得分范围: {top_50['TOPSIS综合得分'].min():.4f} - {top_50['TOPSIS综合得分'].max():.4f}")
print(f"所有供应商TOPSIS得分范围: {results['TOPSIS综合得分'].min():.4f} - {results['TOPSIS综合得分'].max():.4f}")

# 保存结果到Excel文件
results_sorted = results.sort_values('TOPSIS综合得分', ascending=False)
results_sorted.to_excel('供应商TOPSIS评价结果.xlsx', index=False)
print(f"\n完整评价结果已保存到 '供应商TOPSIS评价结果.xlsx'")

TOPSIS模型分析结果 - 排名前10的供应商:
    供应商ID 材料分类  TOPSIS综合得分  排名
228  S229    A    0.792168   1
139  S140    B    0.729009   2
360  S361    C    0.702825   3
200  S201    A    0.589951   4
107  S108    B    0.553579   5
138  S139    B    0.481661   6
150  S151    C    0.478245   7
281  S282    A    0.419742   8
339  S340    B    0.407429   9
274  S275    A    0.402834  10

前50家最重要供应商材料分类分布:
材料分类
C    20
A    15
B    15
Name: count, dtype: int64

前50家供应商TOPSIS得分范围: 0.1742 - 0.7922
所有供应商TOPSIS得分范围: 0.0498 - 0.7922

完整评价结果已保存到 '供应商TOPSIS评价结果.xlsx'


第二题

In [3]:
import pandas as pd
import numpy as np

# 重新读取数据
df = pd.read_excel('附件1.xlsx')

# 提取周数据列
week_columns = [col for col in df.columns if col.startswith('W')]
df_week = df[week_columns].copy()
df_week = df_week.apply(pd.to_numeric, errors='coerce')

# 按24周划分周期（10个周期）
n_periods = 10
period_weeks = 24
total_weeks = n_periods * period_weeks

print(f"周期划分: {n_periods}个周期 × {period_weeks}周 = {total_weeks}周")

# 为每个供应商计算期望供货量
expected_supply = []

for supplier_idx in range(len(df)):
    supplier_id = df.iloc[supplier_idx]['供应商ID']
    material_type = df.iloc[supplier_idx]['材料分类']

    # 获取该供应商的240周数据
    supplier_data = df_week.iloc[supplier_idx].values

    # 按周期分组（10个周期，每个周期24周）
    periods_data = []
    for period in range(n_periods):
        start_idx = period * period_weeks
        end_idx = start_idx + period_weeks
        period_data = supplier_data[start_idx:end_idx]
        periods_data.append(period_data)

    # 计算期望供货量
    weekly_expected = []
    for week_in_period in range(period_weeks):
        # 提取10个周期中对应周次的数据
        week_data = [periods_data[period][week_in_period] for period in range(n_periods)]

        # 移除NaN值
        week_data_clean = [x for x in week_data if pd.notna(x)]

        if not week_data_clean:
            weekly_expected.append(0)
            continue

        # 计算该周次的均值
        week_mean = np.mean(week_data_clean)

        # 筛选条件：供货量/订货量 > 90%（假设订货量=供货量，所以都满足）且 供货量 ≤ 3倍均值
        # 由于假设订货量=供货量，所以供货量/订货量 = 100% > 90%
        valid_data = [x for x in week_data_clean if x <= 3 * week_mean]

        if not valid_data:
            weekly_expected.append(0)
        else:
            # 取最大值作为期望供货量
            weekly_expected.append(max(valid_data))

    # 存储结果
    expected_supply.append({
        '供应商ID': supplier_id,
        '材料分类': material_type,
        **{f'未来第{i + 1}周': weekly_expected[i] for i in range(period_weeks)}
    })

# 创建结果DataFrame
result_df = pd.DataFrame(expected_supply)

print(f"\n期望供货量计算完成，共{len(result_df)}个供应商")
print("\n前5个供应商的期望供货量（前10周）:")
display_cols = ['供应商ID', '材料分类'] + [f'未来第{i + 1}周' for i in range(10)]
print(result_df[display_cols].head())

# 保存结果
result_df.to_excel('期望供货量预测结果.xlsx', index=False)
print(f"\n结果已保存到: 期望供货量预测结果.xlsx")

# 统计信息
print("\n期望供货量统计信息:")
future_weeks = [col for col in result_df.columns if col.startswith('未来第')]
future_data = result_df[future_weeks]
print(f"各周期望供货量平均值:")
for week in future_weeks[:10]:  # 显示前10周
    print(f"{week}: {future_data[week].mean():.2f}")

周期划分: 10个周期 × 24周 = 240周

期望供货量计算完成，共403个供应商

前5个供应商的期望供货量（前10周）:
  供应商ID 材料分类  未来第1周  未来第2周  未来第3周  未来第4周  未来第5周  未来第6周  未来第7周  未来第8周  未来第9周  \
0  S001    B    0.0    0.0    1.0    1.0    0.0    0.0    0.0    1.0    1.0   
1  S002    A    1.0    1.0    1.0    2.0    1.0    0.0    0.0    1.0    1.0   
2  S003    C   10.0    3.0    0.0    0.0    1.0    0.0    4.0   70.0  440.0   
3  S004    B    1.0    1.0    1.0    1.0    1.0    1.0    0.0    1.0    1.0   
4  S005    A   30.0    0.0    1.0   60.0    1.0   70.0   60.0   60.0    0.0   

   未来第10周  
0     1.0  
1     3.0  
2   380.0  
3     1.0  
4     1.0  

结果已保存到: 期望供货量预测结果.xlsx

期望供货量统计信息:
各周期望供货量平均值:
未来第1周: 137.75
未来第2周: 59.49
未来第3周: 70.49
未来第4周: 51.92
未来第5周: 104.21
未来第6周: 68.22
未来第7周: 49.08
未来第8周: 61.41
未来第9周: 63.58
未来第10周: 58.89


In [5]:
import pandas as pd
import numpy as np
import json

# 读取结果数据
result_df = pd.read_excel('期望供货量预测结果.xlsx')

# 分析结果数据
print("=== 期望供货量分析 ===")

# 1. 按材料分类统计
material_stats = {}
future_weeks = [col for col in result_df.columns if col.startswith('未来第')]

for material in ['A', 'B', 'C']:
    material_data = result_df[result_df['材料分类'] == material]
    weekly_means = [float(material_data[week].mean()) for week in future_weeks]

    material_stats[material] = {
        '供应商数量': int(len(material_data)),
        '平均期望供货量': float(np.mean(weekly_means)),
        '周平均供货量': weekly_means
    }

# 2. 周度趋势分析
weekly_totals = [float(result_df[week].sum()) for week in future_weeks]
weekly_avgs = [float(result_df[week].mean()) for week in future_weeks]

# 3. 供应商供货能力分析
supplier_totals = result_df[future_weeks].sum(axis=1)
supplier_totals_list = [float(x) for x in supplier_totals]

# 4. 供应商分布数据
supplier_ranges = ['0-100', '101-500', '501-1000', '1001-5000', '5000+']
supplier_distribution = [
    int(len(supplier_totals[supplier_totals <= 100])),
    int(len(supplier_totals[(supplier_totals > 100) & (supplier_totals <= 500)])),
    int(len(supplier_totals[(supplier_totals > 500) & (supplier_totals <= 1000)])),
    int(len(supplier_totals[(supplier_totals > 1000) & (supplier_totals <= 5000)])),
    int(len(supplier_totals[supplier_totals > 5000]))
]

# 5. 准备样例数据（前10个供应商的前10周数据）
sample_data = []
for i in range(min(10, len(result_df))):
    row = result_df.iloc[i]
    sample_record = {
        '供应商ID': str(row['供应商ID']),
        '材料分类': str(row['材料分类']),
        '周数据': [float(row[f'未来第{j + 1}周']) for j in range(10)]
    }
    sample_data.append(sample_record)

# 6. 准备可视化数据
viz_data = {
    'sample_data': sample_data,
    'weekly_trend': {
        'weeks': [f'第{i + 1}周' for i in range(24)],
        'total_supply': weekly_totals,
        'avg_supply': weekly_avgs
    },
    'material_comparison': {
        'materials': list(material_stats.keys()),
        'avg_supply': [material_stats[m]['平均期望供货量'] for m in material_stats.keys()],
        'supplier_count': [material_stats[m]['供应商数量'] for m in material_stats.keys()],
        'weekly_trends': {
            m: material_stats[m]['周平均供货量'] for m in material_stats.keys()
        }
    },
    'supplier_distribution': {
        'ranges': supplier_ranges,
        'counts': supplier_distribution
    },
    'summary_stats': {
        'total_suppliers': int(len(result_df)),
        'avg_weekly_supply': float(np.mean(weekly_totals)),
        'max_weekly_supply': float(max(weekly_totals)),
        'min_weekly_supply': float(min(weekly_totals))
    }
}

print("样例数据（前3个供应商）:")
for i, record in enumerate(sample_data[:3]):
    print(f"{i + 1}. {record['供应商ID']} ({record['材料分类']}): {record['周数据'][:5]}...")

print(f"\n周度趋势: 第1周{weekly_totals[0]:.0f}, 第24周{weekly_totals[-1]:.0f}")
print(
    f"材料分类分布: A类{material_stats['A']['供应商数量']}家, B类{material_stats['B']['供应商数量']}家, C类{material_stats['C']['供应商数量']}家")
print(f"供应商分布: {dict(zip(supplier_ranges, supplier_distribution))}")
print(
    f"总体统计: 平均周供货量{viz_data['summary_stats']['avg_weekly_supply']:.0f}, 最高{viz_data['summary_stats']['max_weekly_supply']:.0f}")



=== 期望供货量分析 ===
样例数据（前3个供应商）:
1. S001 (B): [0.0, 0.0, 1.0, 1.0, 0.0]...
2. S002 (A): [1.0, 1.0, 1.0, 2.0, 1.0]...
3. S003 (C): [10.0, 3.0, 0.0, 0.0, 1.0]...

周度趋势: 第1周55512, 第24周20993
材料分类分布: A类146家, B类134家, C类122家
供应商分布: {'0-100': 334, '101-500': 10, '501-1000': 6, '1001-5000': 23, '5000+': 30}
总体统计: 平均周供货量27423, 最高55512


In [7]:
import pandas as pd
import numpy as np

# 重新读取数据
df = pd.read_excel('期望供货量预测结果.xlsx', sheet_name=0)
df_clean = df.dropna(subset=['供应商ID']).copy()

# 换算系数
conversion_factors = {'A': 1 / 0.6, 'B': 1 / 0.66, 'C': 1 / 0.72}
week_columns = [col for col in df_clean.columns if '未来第' in col]

print("换算系数：")
for material, factor in conversion_factors.items():
    print(f"1{material} = {factor:.4f}产品")

# 重新计算等效供货量
df_equivalent = df_clean[['供应商ID', '材料分类']].copy()

for week in week_columns:
    # 使用向量化操作计算等效供货量
    mask_a = df_clean['材料分类'] == 'A'
    mask_b = df_clean['材料分类'] == 'B'
    mask_c = df_clean['材料分类'] == 'C'

    equivalent_col = f'等效_{week}'
    df_equivalent[equivalent_col] = 0

    df_equivalent.loc[mask_a, equivalent_col] = df_clean.loc[mask_a, week] * conversion_factors['A']
    df_equivalent.loc[mask_b, equivalent_col] = df_clean.loc[mask_b, week] * conversion_factors['B']
    df_equivalent.loc[mask_c, equivalent_col] = df_clean.loc[mask_c, week] * conversion_factors['C']

# 验证计算结果
print(f"\n第1周数据验证：")
for material in ['A', 'B', 'C']:
    mask = df_equivalent['材料分类'] == material
    raw_qty = df_clean.loc[mask, '未来第1周'].sum()
    converted_qty = df_equivalent.loc[mask, '等效_未来第1周'].sum()
    print(f"{material}材料: 原始数量 {raw_qty}, 等效产品数量 {converted_qty:.2f}")

# 计算各材料类型对等效供货量的贡献
material_contribution = {}
for material in ['A', 'B', 'C']:
    mask = df_equivalent['材料分类'] == material
    contribution = {}
    for week in week_columns:
        equivalent_col = f'等效_{week}'
        contribution[week] = df_equivalent.loc[mask, equivalent_col].sum()
    material_contribution[material] = contribution

print(f"\n各材料类型总贡献：")
total_by_material = {}
for material in ['A', 'B', 'C']:
    total = sum(material_contribution[material].values())
    total_by_material[material] = total
    print(f"{material}: {total:.2f}")

# 计算每周总等效供货量和过剩量
capacity = 28200
weekly_summary = []

for week in week_columns:
    equivalent_col = f'等效_{week}'
    total_equivalent = df_equivalent[equivalent_col].sum()
    surplus = total_equivalent - capacity

    weekly_summary.append({
        'week': int(week.replace('未来第', '').replace('周', '')),
        'total_equivalent': round(total_equivalent, 2),
        'surplus': round(surplus, 2),
        'a_contribution': round(material_contribution['A'][week], 2),
        'b_contribution': round(material_contribution['B'][week], 2),
        'c_contribution': round(material_contribution['C'][week], 2)
    })

print(f"\n前5周汇总：")
for summary in weekly_summary[:5]:
    print(f"第{summary['week']}周: 总等效供货量 {summary['total_equivalent']}, 过剩 {summary['surplus']}")

# 计算整体统计
total_surplus = sum([s['surplus'] for s in weekly_summary])
avg_surplus = total_surplus / len(weekly_summary)
max_surplus = max(weekly_summary, key=lambda x: x['surplus'])
min_surplus = min(weekly_summary, key=lambda x: x['surplus'])

print(f"\n整体统计：")
print(f"总过剩量: {total_surplus:.2f}")
print(f"平均每周过剩: {avg_surplus:.2f}")
print(f"过剩最多周: 第{max_surplus['week']}周 ({max_surplus['surplus']:.2f})")
print(f"过剩最少周: 第{min_surplus['week']}周 ({min_surplus['surplus']:.2f})")



# 保存等效期望供货量到新的Excel文件
output_file = '等效期望供货量结果.xlsx'
df_equivalent.to_excel(output_file, index=False)
print(f"等效期望供货量已成功保存到 {output_file}")


换算系数：
1A = 1.6667产品
1B = 1.5152产品
1C = 1.3889产品

第1周数据验证：
A材料: 原始数量 10824, 等效产品数量 18040.00
B材料: 原始数量 36875, 等效产品数量 55871.21
C材料: 原始数量 7813, 等效产品数量 10851.39

各材料类型总贡献：
A: 376393.33
B: 336518.18
C: 291950.00

前5周汇总：
第1周: 总等效供货量 84762.6, 过剩 56562.6
第2周: 总等效供货量 36300.05, 过剩 8100.05
第3周: 总等效供货量 42880.48, 过剩 14680.48
第4周: 总等效供货量 31830.83, 过剩 3630.83
第5周: 总等效供货量 63754.19, 过剩 35554.19

整体统计：
总过剩量: 328061.49
平均每周过剩: 13669.23
过剩最多周: 第1周 (56562.60)
过剩最少周: 第7周 (2082.90)
等效期望供货量已成功保存到 等效期望供货量结果.xlsx


In [10]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm

# 读取数据
df_supply = pd.read_excel('等效期望供货量结果.xlsx')
df_topsis = pd.read_excel('供应商TOPSIS评价结果精简版.xlsx')

# 合并供应商数据
df_combined = pd.merge(df_supply, df_topsis, on='供应商ID')

# 关键参数
weekly_demand = 28200  # 周产能需求28200立方米
min_inventory = weekly_demand * 2  # 最小库存要求两周产能

# 准备数据
supply_weeks = [col for col in df_combined.columns if col.startswith('等效_未来第')]
n_weeks = len(supply_weeks)
n_suppliers = len(df_combined)

# 提取供应商供货数据和TOPSIS评分
supply_matrix = df_combined[supply_weeks].values  # 形状: (n_suppliers, n_weeks)
topsis_scores = df_combined['TOPSIS综合得分'].values

print(f"简化模型数据准备完成:")
print(f"- 供应商数量: {n_suppliers}")
print(f"- 计划周数: {n_weeks}")
print(f"- 周产能需求: {weekly_demand}立方米")
print(f"- 最小库存要求: {min_inventory}立方米")

# 简化版遗传算法参数
population_size = 100
generations = 200
mutation_rate = 0.05
elitism_rate = 0.1


# 初始化种群 - 基于TOPSIS评分的启发式初始化
def initialize_population(size):
    population = []

    # 按TOPSIS评分排序
    sorted_indices = np.argsort(topsis_scores)[::-1]

    for _ in range(size):
        # 随机选择不同数量的供应商，偏向于选择评分高的
        num_selected = random.randint(10, 50)
        selected_indices = sorted_indices[:num_selected].copy()

        # 随机替换一些供应商，增加多样性
        if random.random() < 0.3:
            num_replace = random.randint(1, 5)
            replace_indices = random.sample(range(num_selected), num_replace)
            new_indices = random.sample(range(num_selected, n_suppliers), num_replace)
            selected_indices[replace_indices] = new_indices

        chromosome = np.zeros(n_suppliers)
        chromosome[selected_indices] = 1
        population.append(chromosome)

    return population


# 简化版适应度函数 - 不考虑转运商约束
def calculate_fitness(chromosome):
    selected_indices = np.where(chromosome == 1)[0]
    num_selected = len(selected_indices)

    if num_selected == 0:
        return 0

    # 计算每周供应量
    weekly_supply = np.sum(supply_matrix[selected_indices, :], axis=0)

    # 简化计算：假设100%运输效率，不考虑损耗
    # 计算库存动态变化
    inventory = min_inventory  # 初始库存为最小要求
    valid = True

    for week_idx in range(n_weeks):
        # 周需求
        demand = weekly_demand

        # 实际可用量 = 本周供应 + 库存
        available = weekly_supply[week_idx] + inventory

        # 检查是否满足需求
        if available < demand:
            valid = False
            break

        # 更新库存
        inventory = available - demand

        # 检查库存是否低于最小要求
        if inventory < min_inventory:
            valid = False
            break

    if not valid:
        return 0  # 不满足约束条件，适应度为0

    # 适应度 = 1 / (供应商数量) * TOPSIS评分加权
    avg_topsis = np.mean(topsis_scores[selected_indices]) if num_selected > 0 else 0
    return (1.0 / num_selected) * (0.7 + 0.3 * avg_topsis)


# 选择操作
def selection(population, fitness_scores):
    total_fitness = np.sum(fitness_scores)
    if total_fitness == 0:
        return random.sample(population, len(population))

    probabilities = fitness_scores / total_fitness
    selected = []

    # 精英保留
    elite_size = int(elitism_rate * len(population))
    elite_indices = np.argsort(fitness_scores)[::-1][:elite_size]
    for idx in elite_indices:
        selected.append(population[idx])

    # 轮盘赌选择
    for _ in range(len(population) - elite_size):
        selected_idx = np.random.choice(len(population), p=probabilities)
        selected.append(population[selected_idx])

    return selected


# 交叉操作 - 单点交叉
def crossover(parent1, parent2):
    if random.random() < 0.8:  # 交叉概率80%
        crossover_point = random.randint(1, len(parent1) - 1)
        child1 = np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])
        child2 = np.concatenate([parent2[:crossover_point], parent1[crossover_point:]])
        return child1, child2
    return parent1.copy(), parent2.copy()


# 变异操作
def mutate(chromosome):
    for i in range(len(chromosome)):
        if random.random() < mutation_rate:
            chromosome[i] = 1 - chromosome[i]  # 翻转位
    return chromosome


# 运行简化版遗传算法
def run_simplified_ga():
    print("\n开始运行简化版遗传算法...")
    population = initialize_population(population_size)

    best_fitness_history = []
    best_solution = None
    best_fitness = 0

    for generation in tqdm(range(generations), desc="进化中"):
        # 计算适应度
        fitness_scores = np.array([calculate_fitness(chrom) for chrom in population])

        # 记录最佳解
        current_best_idx = np.argmax(fitness_scores)
        current_best_fitness = fitness_scores[current_best_idx]
        current_best_chrom = population[current_best_idx]

        best_fitness_history.append(current_best_fitness)

        if current_best_fitness > best_fitness:
            best_fitness = current_best_fitness
            best_solution = current_best_chrom.copy()

        # 选择
        selected_population = selection(population, fitness_scores)

        # 交叉和变异
        next_population = []
        for i in range(0, len(selected_population), 2):
            if i + 1 < len(selected_population):
                parent1 = selected_population[i]
                parent2 = selected_population[i + 1]
                child1, child2 = crossover(parent1, parent2)
                child1 = mutate(child1)
                child2 = mutate(child2)
                next_population.extend([child1, child2])
            else:
                next_population.append(selected_population[i])

        population = next_population[:population_size]

        # 打印当前代信息
        if generation % 20 == 0:
            best_supplier_count = np.sum(best_solution) if best_solution is not None else 0
            print(f"第{generation}代: 最佳适应度={current_best_fitness:.6f}, 最佳供应商数量={best_supplier_count}")

    return best_solution, best_fitness, best_fitness_history


# 运行简化版遗传算法
best_solution, best_fitness, fitness_history = run_simplified_ga()

# 分析结果
visualization_data = None  # 初始化变量

if best_solution is not None:
    selected_indices = np.where(best_solution == 1)[0]
    num_selected = len(selected_indices)

    print(f"\n简化版遗传算法优化完成!")
    print(f"最佳供应商数量: {num_selected}")
    print(f"最佳适应度: {best_fitness:.6f}")

    # 验证最佳解
    print("\n验证最佳解...")
    selected_suppliers = df_combined.iloc[selected_indices]

    # 计算每周供应量
    weekly_supply = np.sum(supply_matrix[selected_indices, :], axis=0)

    # 验证库存动态
    inventory = min_inventory
    valid = True
    weekly_inventory = []

    for week_idx in range(n_weeks):
        available = weekly_supply[week_idx] + inventory
        if available < weekly_demand:
            valid = False
        inventory = available - weekly_demand
        if inventory < min_inventory:
            valid = False
        weekly_inventory.append(inventory)

    print(f"解决方案有效性: {'有效' if valid else '无效'}")
    print(f"材料分类分布: {selected_suppliers['材料分类'].value_counts().to_dict()}")
    print(
        f"TOPSIS得分范围: {selected_suppliers['TOPSIS综合得分'].min():.4f} - {selected_suppliers['TOPSIS综合得分'].max():.4f}")
    print(f"平均TOPSIS得分: {selected_suppliers['TOPSIS综合得分'].mean():.4f}")

    # 详细供需分析
    print(f"\n每周供需分析:")
    for i in range(n_weeks):
        print(
            f"第{i + 1}周: 供应量={weekly_supply[i]:.0f}立方米, 需求={weekly_demand}立方米, 库存={weekly_inventory[i]:.0f}立方米")

    # 保存结果
    result_data = {
        '供应商ID': selected_suppliers['供应商ID'],
        '材料分类': selected_suppliers['材料分类'],
        'TOPSIS综合得分': selected_suppliers['TOPSIS综合得分'],
        '排名': selected_suppliers['排名']
    }
    df_result = pd.DataFrame(result_data)
    df_result.to_excel('最佳供应商选择结果_简化模型.xlsx', index=False)
    print(f"\n结果已保存到: 最佳供应商选择结果_简化模型.xlsx")

    # 生成可视化数据
    visualization_data = {
        'best_supplier_count': int(num_selected),
        'material_distribution': selected_suppliers['材料分类'].value_counts().to_dict(),
        'topsis_stats': {
            'min': float(selected_suppliers['TOPSIS综合得分'].min()),
            'max': float(selected_suppliers['TOPSIS综合得分'].max()),
            'mean': float(selected_suppliers['TOPSIS综合得分'].mean())
        },
        'weekly_supply': weekly_supply.tolist(),
        'weekly_demand': weekly_demand,
        'weekly_inventory': weekly_inventory,
        'fitness_history': fitness_history,
        'selected_suppliers_sample': selected_suppliers.head(10)[
            ['供应商ID', '材料分类', 'TOPSIS综合得分', '排名']].to_dict('records')
    }
else:
    print("未找到有效的解决方案")


简化模型数据准备完成:
- 供应商数量: 402
- 计划周数: 24
- 周产能需求: 28200立方米
- 最小库存要求: 56400立方米

开始运行简化版遗传算法...


进化中:   8%|▊         | 16/200 [00:00<00:01, 159.32it/s]

第0代: 最佳适应度=0.049557, 最佳供应商数量=17.0
第20代: 最佳适应度=0.005131, 最佳供应商数量=17.0


进化中:  34%|███▎      | 67/200 [00:00<00:00, 164.01it/s]

第40代: 最佳适应度=0.004767, 最佳供应商数量=17.0
第60代: 最佳适应度=0.004425, 最佳供应商数量=17.0


...

第120代: 最佳适应度=0.004582, 最佳供应商数量=17.0
第140代: 最佳适应度=0.004414, 最佳供应商数量=17.0


进化中:  95%|█████████▌| 190/200 [00:01<00:00, 171.56it/s]

第160代: 最佳适应度=0.004217, 最佳供应商数量=17.0
第180代: 最佳适应度=0.004610, 最佳供应商数量=17.0


进化中: 100%|██████████| 200/200 [00:01<00:00, 167.24it/s]


简化版遗传算法优化完成!
最佳供应商数量: 17
最佳适应度: 0.049557

验证最佳解...
解决方案有效性: 有效
材料分类分布: {'B': 7, 'A': 6, 'C': 4}
TOPSIS得分范围: 0.3348 - 0.7922
平均TOPSIS得分: 0.4749

每周供需分析:
第1周: 供应量=69185立方米, 需求=28200立方米, 库存=97385立方米
第2周: 供应量=27683立方米, 需求=28200立方米, 库存=96867立方米
第3周: 供应量=32816立方米, 需求=28200立方米, 库存=101484立方米
第4周: 供应量=18158立方米, 需求=28200立方米, 库存=91442立方米
第5周: 供应量=53384立方米, 需求=28200立方米, 库存=116626立方米
第6周: 供应量=19675立方米, 需求=28200立方米, 库存=108101立方米
第7周: 供应量=20992立方米, 需求=28200立方米, 库存=100893立方米
第8周: 供应量=29454立方米, 需求=28200立方米, 库存=102147立方米
第9周: 供应量=22179立方米, 需求=28200立方米, 库存=96126立方米
第10周: 供应量=22692立方米, 需求=28200立方米, 库存=90619立方米
第11周: 供应量=23835立方米, 需求=28200立方米, 库存=86254立方米
第12周: 供应量=23253立方米, 需求=28200立方米, 库存=81307立方米
第13周: 供应量=25269立方米, 需求=28200立方米, 库存=78376立方米
第14周: 供应量=28438立方米, 需求=28200立方米, 库存=78614立方米
第15周: 供应量=23803立方米, 需求=28200立方米, 库存=74217立方米
第16周: 供应量=27358立方米, 需求=28200立方米, 库存=73374立方米
第17周: 供应量=32127立方米, 需求=28200立方米, 库存=77302立方米
第18周: 供应量=23163立方米, 需求=28200立方米, 库存=72265立方米
第19周: 供应量=25519立方米, 需求=28200立方米, 库存=69584